# Multimodalni model: Sentinel-2 + otisci zgrada (zajednicki trening)

Fuzija dogovorena u prepisci sa profesorom: **jedan model koji istovremeno
dobija strukturirane podatke i sliku**. Tri grane:

| grana | ulaz | trup |
|---|---|---|
| satelitska | Sentinel-2 isecak, 6 opsega | ResNet-18 (ImageNet), 512-dim |
| rasterska | otisci zgrada, 2 kanala | ResNet-18 (ImageNet), 512-dim |
| tabelarna | 14 strukturiranih atributa otisaka (iz geometrije) | MLP, 32-dim |

Sve tri reprezentacije se konkateniraju (1056-dim) i idu u zajednicku regresionu
glavu.
Cilj je `log1p(broj_stanovnika)`; trening end-to-end (glava pa fine-tuning oba
trupa). Ista GroupKFold podela po opstinama i isti OOF protokol kao u ostalim
notebucima, pa su rezultati direktno uporedivi (i sa stacking fuzijom).

## Instalacija

In [ ]:
%pip install -q timm mlflow
try:
    dbutils.library.restartPython()
except NameError:
    pass

## Konfiguracija

In [ ]:
# Postavke hiperparametara
import sys, os
# koren repoa (sadrzi core/): penji se uz stablo od radnog direktorijuma -
# radi na Databricksu pod bilo kojim nalogom i u lokalnom klonu
_koren = os.getcwd()
while not os.path.isdir(os.path.join(_koren, "core")) and os.path.dirname(_koren) != _koren:
    _koren = os.path.dirname(_koren)
assert os.path.isdir(os.path.join(_koren, "core")), "nema core/ - kloniraj ceo repo, ne samo notebook"
if _koren not in sys.path:
    sys.path.insert(0, _koren)

import glob
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm, mlflow
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from core import (
    seed_everything, dvofazni_trening,
    stats_po_opsegu, NW, seed_worker,
    napravi_foldove, oof_metrics, cv_summary_figure,
    podesi_mlflow, izlazni_dir, sacuvaj_oof,
)

# podaci na Databricks UC Volume-u (isti "data" folder kao ostali pristupi)
BASE = "/Volumes/katalog/deep_learning/raw_data/data"
CUT_SAT = BASE + "/cutouts"
CUT_FP  = BASE + "/footprint_cutouts"
OUT_DIR = izlazni_dir()   # tezine modela i OOF parquet (UC Volume, lokalno "out/")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# svi hiperparametri na jednom mestu (jedini izvor istine)
CFG = {
    "sat_bands": 6,                 # Sentinel-2 opsega (satelitska grana)
    "fp_bands": 2,                  # kanali otisaka: pokrivenost, zapreminska gustina
    "tab_hidden": 32,               # izlaz tabelarne grane (MLP nad strukturiranim atributima)
    "tab_dropout": 0.15,
    "img_px": 224,
    "embed_hidden": 256,            # skriveni sloj zajednicke glave (1024 -> 256 -> 1)
    "dropout": 0.2,
    "epochs_head": 3,
    "epochs_finetune": 40,
    "batch_size": 48,               # dva ResNet-18 trupa -> nesto manji batch nego kod jednog
    "head_lr": 1e-3,
    "finetune_lr": 3e-4,
    "seed": 42,                     # za reproduktivnost (random/numpy/torch/cuda + DataLoader radnici)
}
seed_everything(CFG["seed"])
print("device:", DEVICE, "| sat cutouts:", len(os.listdir(CUT_SAT)), "| footprint cutouts:", len(os.listdir(CUT_FP)))

## Podaci i podela (presek naselja sa oba ulaza)

In [ ]:
# Ucitavanje podataka / podesavanje CV
labele = pd.read_parquet(BASE + "/naselje_table.parquet")[
    ["naselje_maticni_broj", "opstina_maticni_broj", "pop"]]

def putanje(folder):
    t = pd.DataFrame({"path": glob.glob(folder + "/*.npy")})
    t["naselje_maticni_broj"] = t.path.map(lambda f: int(os.path.splitext(os.path.basename(f))[0]))
    return t

sat = putanje(CUT_SAT).rename(columns={"path": "path_sat"})
fp  = putanje(CUT_FP).rename(columns={"path": "path_fp"})
df = (
    sat.merge(fp, on="naselje_maticni_broj", how="inner")   # samo naselja sa OBA ulaza
    .merge(labele, on="naselje_maticni_broj", how="inner")
)
df["y"] = np.log1p(df["pop"]).astype("float32")

# tabelarna grana: isti strukturirani atributi kao u 03_footprint_train
# svi atributi su iz geometrije. Overture opisne kolone (num_floors 1.17%,
# height 0.04%, subtype/class ~10.8%) se namerno ne koriste: popunjenost je
# neravnomerna po okruzima pa bi bila proksi za gustinu mapiranja, ne za
# izgradjenost. Detalji u scripts/footprint/per_naselje.py.
LOG_ATRIBUTI = ["n_buildings", "roof_area_m2", "mean_bsize", "median_bsize", "std_bsize",
                "p90_bsize", "mean_nn_dist", "median_nn_dist", "mean_n_50m",
                "building_density", "area_km2"]
LIN_ATRIBUTI = ["mean_compact", "udeo_velikih", "built_fraction"]
ATRIBUTI = LOG_ATRIBUTI + LIN_ATRIBUTI

atr = pd.read_parquet(BASE + "/naselje_footprints.parquet")[["naselje_maticni_broj", *ATRIBUTI]]
for c in LOG_ATRIBUTI:
    atr[c] = np.log1p(atr[c].clip(lower=0))
df = df.merge(atr, on="naselje_maticni_broj", how="inner")   # sva tri modaliteta

print(f"sat {len(sat)} | footprint {len(fp)} | presek (sva tri ulaza + labela) {len(df)}")

FOLDS = napravi_foldove(df)
N_FOLDS = len(FOLDS)
broj_opstina = df["opstina_maticni_broj"].nunique()
print(f"uzoraka {len(df)} | opstina {broj_opstina} | foldova {N_FOLDS}")
for i, (t, v) in enumerate(FOLDS):
    print(f"  fold {i}: trening {len(t)} / val {len(v)} naselja ({v['opstina_maticni_broj'].nunique()} opstina)")

## Normalizacija i dataset

In [ ]:
# Dataset sa dva ulaza
# stats_po_opsegu, NW, seed_worker su uvezeni iz core.data;
# NaseljaMM je multimodal-specifican (dva .npy ulaza, ista prostorna augmentacija na oba).

class NaseljaMM(Dataset):
    """Trojka (satelitski cutout, footprint raster, strukturirani atributi)
    istog naselja + log1p(pop).

    Augmentacija (flip/rotacija) se izvlaci jednom i primenjuje identicno na
    oba rastera — prostorno su poravnati pa transformacije moraju biti iste.
    Atributi su skalari, njih augmentacija ne dira.
    """

    def __init__(self, frame, mean_s, std_s, mean_f, std_f, skaler, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.mean_s, self.std_s = mean_s, std_s
        self.mean_f, self.std_f = mean_f, std_f
        self.tab = skaler.transform(self.frame[ATRIBUTI].values).astype("float32")
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        red = self.frame.iloc[i]
        xs = (np.load(red.path_sat).astype("float32") - self.mean_s[0]) / self.std_s[0]
        xf = (np.load(red.path_fp).astype("float32") - self.mean_f[0]) / self.std_f[0]
        if self.augment:
            if np.random.rand() < 0.5:
                xs = xs[:, :, ::-1]; xf = xf[:, :, ::-1]
            if np.random.rand() < 0.5:
                xs = xs[:, ::-1, :]; xf = xf[:, ::-1, :]
            k = np.random.randint(4)
            xs = np.rot90(xs, k, axes=(1, 2)); xf = np.rot90(xf, k, axes=(1, 2))
        return (
            torch.from_numpy(np.ascontiguousarray(xs)),
            torch.from_numpy(np.ascontiguousarray(xf)),
            torch.from_numpy(self.tab[i]),
            torch.tensor([red.y], dtype=torch.float32),
        )

def napravi_loadere_mm(train_frame, val_frame):
    """Vrati (train_dl, val_dl); normalizacija po modalitetu iz trening skupa ovog folda."""
    mean_s, std_s = stats_po_opsegu(train_frame.path_sat.tolist())
    mean_f, std_f = stats_po_opsegu(train_frame.path_fp.tolist())
    # atributi: standardizacija fitovana SAMO na trening foldu, isto pravilo
    # kao stats_po_opsegu za rastere
    skaler = StandardScaler().fit(train_frame[ATRIBUTI].values)
    gen = torch.Generator().manual_seed(CFG["seed"])
    _sw = (lambda wid: seed_worker(wid, CFG["seed"])) if NW else None
    tdl = DataLoader(NaseljaMM(train_frame, mean_s, std_s, mean_f, std_f, skaler, augment=True),
                     batch_size=CFG["batch_size"], shuffle=True,
                     # batch od tacno 1 uzorka rusi BatchNorm u trening modu
                     drop_last=len(train_frame) % CFG["batch_size"] == 1,
                     generator=gen, worker_init_fn=_sw,
                     num_workers=NW, pin_memory=True, persistent_workers=NW > 0,
                     prefetch_factor=4 if NW else None)
    vdl = DataLoader(NaseljaMM(val_frame, mean_s, std_s, mean_f, std_f, skaler),
                     batch_size=CFG["batch_size"],
                     num_workers=NW, pin_memory=True, persistent_workers=NW > 0,
                     prefetch_factor=4 if NW else None)
    return tdl, vdl

## Model (dve grane + zajednicka glava)

In [ ]:
# Multimodalni model
class MultimodalniModel(nn.Module):
    """Tri grane: dva ResNet-18 trupa (num_classes=0 -> 512-dim embedding) i MLP
    nad strukturiranim atributima; konkatenacija sve tri ide u zajednicku
    regresionu glavu.

    Parametri glave i tabelarne grane pocinju sa "head" — dvofazni_trening(
    head_prefix="head") u fazi 1 trenira samo njih, oba trupa zamrznuta. Tabelarna
    grana je mala i uci se iz nule pa nema razloga da bude zamrznuta."""

    def __init__(self):
        super().__init__()
        self.sat = timm.create_model("resnet18", pretrained=True,
                                     in_chans=CFG["sat_bands"], num_classes=0)
        self.fp  = timm.create_model("resnet18", pretrained=True,
                                     in_chans=CFG["fp_bands"], num_classes=0)
        self.head_tab = nn.Sequential(
            nn.Linear(len(ATRIBUTI), CFG["tab_hidden"]),
            nn.BatchNorm1d(CFG["tab_hidden"]),
            nn.ReLU(),
            nn.Dropout(CFG["tab_dropout"]),
        )
        d = self.sat.num_features + self.fp.num_features + CFG["tab_hidden"]
        self.head = nn.Sequential(
            nn.Linear(d, CFG["embed_hidden"]),
            nn.ReLU(),
            nn.Dropout(CFG["dropout"]),
            nn.Linear(CFG["embed_hidden"], 1),
        )

    def forward(self, xs, xf, xt):
        return self.head(torch.cat([self.sat(xs), self.fp(xf), self.head_tab(xt)], dim=1))


loss_fn = nn.HuberLoss()
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def prodji_mm(net, loader, treniraj, optim=None, freeze_bn=False):
    """Jedan prolaz; kao core.train.prodji ali sa tri ulaza po uzorku."""
    net.train(treniraj)
    if freeze_bn:                                   # faza 1: trupovi zamrznuti -> ne azuriraj BN statistiku
        for m in net.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()
    ukupno, P, Y = 0.0, [], []
    for xs, xf, xt, y in loader:
        xs = xs.to(DEVICE, non_blocking=True)
        xf = xf.to(DEVICE, non_blocking=True)
        xt = xt.to(DEVICE, non_blocking=True)
        y  = y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(treniraj), torch.autocast("cuda", enabled=use_amp):
            out  = net(xs, xf, xt)
            loss = loss_fn(out, y)
        if treniraj:
            optim.zero_grad(); scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
        ukupno += loss.item() * len(y)
        P.append(out.detach().float().cpu().numpy()); Y.append(y.cpu().numpy())
    return ukupno / len(loader.dataset), np.concatenate(P).ravel(), np.concatenate(Y).ravel()

## Trening (sa MLflow pracenjem)

In [ ]:
# Trening + evaluacija (5-struka CV)
def treniraj_fold(train_frame, val_frame):
    """Istrenira jedan fold (glava pa fine-tuning oba trupa) i vrati
    (best_state, best_val_r2, oof_pred_pop, net)."""
    train_dl, val_dl = napravi_loadere_mm(train_frame, val_frame)
    net = MultimodalniModel().to(DEVICE)

    def epoha(opt, korak, freeze_bn=False):
        tl, _, _ = prodji_mm(net, train_dl, True, opt, freeze_bn=freeze_bn)
        vl, P, Y = prodji_mm(net, val_dl, False)
        r2 = r2_score(Y, P)
        mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=korak)
        return r2

    best_r2, best_state = dvofazni_trening(
        net, epoha,
        CFG["epochs_head"], CFG["epochs_finetune"],
        CFG["head_lr"],     CFG["finetune_lr"],
        head_prefix="head",
    )
    net.load_state_dict(best_state)     # najbolja tezina po validaciji ovog folda
    _, P, _ = prodji_mm(net, val_dl, False)
    oof_pred_pop = np.clip(np.expm1(P), 0, None)   # OOF predikcija u populaciji
    return best_state, best_r2, oof_pred_pop, net

## Evaluacija

In [ ]:
# Pokreni CV
def run():
    """Puna k-struka GroupKFold CV (multimodalni pristup)."""
    podesi_mlflow()   # Databricks workspace; lokalno mlruns/, ili Databricks preko env varijabli
    oof = pd.Series(np.nan, index=df.naselje_maticni_broj.values, dtype="float32")
    fold_r2 = []

    with mlflow.start_run(run_name=f"multimodal-cv{len(FOLDS)}"):
        mlflow.log_params(CFG)
        mlflow.log_params({
            "pristup": "multimodal",
            "backbone": "2x resnet18 (sat 6ch + footprint 2ch) + MLP nad atributima, concat 1056 -> head",
            "n_atributa": len(ATRIBUTI),
            "pretrained": True,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "loss": "HuberLoss",
            "target": "log1p(pop)",
            "cv": f"GroupKFold(opstina) x{len(FOLDS)}",
            "n_uzoraka": len(df),
            "n_opstina": int(broj_opstina),
        })

        for fold, (train_frame, val_frame) in enumerate(FOLDS):
            with mlflow.start_run(run_name=f"multimodal-fold{fold}", nested=True):
                mlflow.log_params({**CFG, "fold": fold,
                                   "n_train": len(train_frame), "n_val": len(val_frame)})
                best_state, best_r2, oof_pred_pop, net = treniraj_fold(train_frame, val_frame)
                mlflow.log_metric("best_val_r2", best_r2)
                oof.loc[val_frame.naselje_maticni_broj.values] = oof_pred_pop
                put = f"{OUT_DIR}/multimodal_fold{fold}.pt"
                torch.save(best_state, put); mlflow.log_artifact(put)
                mlflow.pytorch.log_model(
                    net,
                    name=f"model_multimodal_fold{fold}",
                    serialization_format="pickle"
                )
                fold_r2.append(best_r2)
                print(f"[fold {fold}] best val R2 {best_r2:.3f}")

        # === agregacija preko svih foldova (OOF: svako naselje predvidjeno tacno jednom) ===
        oof_pred = oof.loc[df.naselje_maticni_broj.values].values.astype("float32")
        stvarno  = df["pop"].values.astype("float32")
        agg = {
            "cv_mean_val_r2": float(np.mean(fold_r2)),
            "cv_std_val_r2":  float(np.std(fold_r2)),
            **oof_metrics(stvarno, oof_pred, df),
        }
        mlflow.log_metrics(agg)
        put_oof = sacuvaj_oof(df, oof_pred, "multimodal", OUT_DIR)   # i ulaz za stacking poredjenje
        mlflow.log_artifact(put_oof)
        fig = cv_summary_figure(fold_r2, agg, stvarno, oof_pred, df, label="multimodal")
        plt.show()
        mlflow.log_figure(fig, "cv_evaluacija_multimodal.png")

    print(f"[multimodal] CV R2 {agg['cv_mean_val_r2']:.3f} ± {agg['cv_std_val_r2']:.3f}"
          f" | OOF R2(log) {agg['oof_r2_log']:.3f} | medAPE {agg.get('oof_medape', float('nan')):.2f}"
          f" | wMAPE {agg.get('oof_wmape', float('nan')):.2f} | bias {agg.get('oof_bias', float('nan')):.2f}"
          f" | opstina R2(log, bez top2) {agg.get('oof_opstina_r2_log_bez_top2', float('nan')):.3f}")
    return {"pristup": "multimodal", **agg}


rezultat = run()
print("\n=== Rezultat (multimodalni model) ===")
display(pd.DataFrame([rezultat]).set_index("pristup"))